# Sprint 2 — Convention-Based Mapping

Spec: [`sprint-2-tasks.md`](../../litemapper/docs/requirements/sprint-2-tasks.md) — 12 tasks (S2-T00..T11) that shipped the property-convention pipeline.

| Convention                              | Task    | Example                                                    |
| --------------------------------------- | ------- | ---------------------------------------------------------- |
| `ExactNameConvention`                   | S2-T01  | `Name` ↔ `Name`                                            |
| `CaseConvention`                        | S2-T02  | `first_name` ↔ `FirstName` (case-insensitive)              |
| `FlatteningConvention`                  | S2-T03  | `Customer.Address.City` → `CustomerAddressCity`            |
| `UnflatteningConvention`                | S2-T04  | `CustomerAddressCity` → `Customer.Address.City` (reverse)  |
| `PrefixDroppingConvention`              | S2-T05  | `dbUserId` → `UserId`                                      |
| `MethodToPropertyConvention`            | S2-T06  | `GetFullName()` → `FullName`                               |
| `AbbreviationConvention`                | S2-T07  | `Qty` ↔ `Quantity`                                         |
| `StructuralSimilarityScorer`            | S2-T08  | fuzzy fallback                                             |
| `NameSuffixTypeConvention`              | S2-T10  | `Order` ↔ `OrderDto` auto-pair                             |

The conventions run in a pipeline at forge time. Their results are **observable** through the `PropertyLink.LinkedBy` field — every link records which convention produced it (useful for debugging later in Sprint 7's `Inspect<S, D>()`).


## Setup


In [2]:
#r "../src/SmartMapp.Net/bin/Release/net10.0/SmartMapp.Net.dll"
using SmartMapp.Net;
using SmartMapp.Net.Abstractions;

static void PrintLinks(ISculptor s)
{
    foreach (var bp in ((ISculptorConfiguration)s).GetAllBlueprints())
    {
        Console.WriteLine($"Blueprint {bp.TypePair.OriginType.Name} → {bp.TypePair.TargetType.Name}:");
        foreach (var link in bp.Links)
            Console.WriteLine($"  {link.TargetMember.Name,-24}  ←  {link.LinkedBy.ConventionName,-20}  ({link.LinkedBy.OriginMemberPath})");
    }
}

Console.WriteLine("Helpers ready.");


Helpers ready.


## 1. `ExactName` — same-name + same-type auto-link

The simplest and highest-confidence convention. Runs first in the pipeline.


In [3]:
public sealed class E1_User    { public int Id { get; init; } public string Name { get; init; } = ""; public string Email { get; init; } = ""; }
public sealed class E1_UserDto { public int Id { get; set; }  public string Name { get; set; } = "";  public string Email { get; set; } = ""; }

var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<E1_User, E1_UserDto>(_ => { }))
    .Forge();

PrintLinks(sculptor);


Blueprint E1_User → E1_UserDto:
  Id                        ←  ExactNameConvention   (Id)
  Name                      ←  ExactNameConvention   (Name)
  Email                     ←  ExactNameConvention   (Email)


## 2. `Flattening` — nested path → flat member

Target members with concatenated ancestor names match against the origin's nested graph. `Customer.Address.City` on the origin flows into `CustomerAddressCity` on the target.


In [4]:
public sealed class E2_Address  { public string City { get; init; } = ""; public string Zip { get; init; } = ""; }
public sealed class E2_Customer { public string Name { get; init; } = ""; public E2_Address Address { get; init; } = new(); }
public sealed class E2_Order    { public int Id { get; init; } public E2_Customer Customer { get; init; } = new(); }

public sealed class E2_OrderFlatDto
{
    public int    Id                 { get; set; }
    public string CustomerName       { get; set; } = "";
    public string CustomerAddressCity{ get; set; } = "";
    public string CustomerAddressZip { get; set; } = "";
}

var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<E2_Order, E2_OrderFlatDto>(_ => { }))
    .Forge();

PrintLinks(sculptor);

var order = new E2_Order { Id = 9, Customer = new E2_Customer { Name = "Ada", Address = new E2_Address { City = "London", Zip = "EC1" } } };
var dto = sculptor.Map<E2_Order, E2_OrderFlatDto>(order);
Console.WriteLine($"\nResult: Id={dto.Id}, CustomerName={dto.CustomerName}, CustomerAddressCity={dto.CustomerAddressCity}, CustomerAddressZip={dto.CustomerAddressZip}");


Blueprint E2_Order → E2_OrderFlatDto:
  Id                        ←  ExactNameConvention   (Id)
  CustomerName              ←  FlatteningConvention  (Customer.Name)
  CustomerAddressCity       ←  FlatteningConvention  (Customer.Address.City)
  CustomerAddressZip        ←  FlatteningConvention  (Customer.Address.Zip)

Result: Id=9, CustomerName=Ada, CustomerAddressCity=London, CustomerAddressZip=EC1


## 3. `Unflattening` — flat member → nested path (reverse direction)

The mirror of flattening. When the target has a nested graph and the origin is flat, the convention splits the flat name against the target's property tree.


In [5]:
public sealed class E3_FlatDto  { public int Id { get; init; } public string CustomerName { get; init; } = ""; public string CustomerAddressCity { get; init; } = ""; }
public sealed class E3_Address  { public string City { get; set; } = ""; }
public sealed class E3_Customer { public string Name { get; set; } = ""; public E3_Address Address { get; set; } = new(); }
public sealed class E3_Order    { public int Id { get; set; } public E3_Customer Customer { get; set; } = new(); }

var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<E3_FlatDto, E3_Order>(_ => { }))
    .Forge();

PrintLinks(sculptor);

var src = new E3_FlatDto { Id = 1, CustomerName = "Grace", CustomerAddressCity = "Arlington" };
var order = sculptor.Map<E3_FlatDto, E3_Order>(src);
Console.WriteLine($"\nResult: Id={order.Id}, Customer.Name={order.Customer.Name}, Customer.Address.City={order.Customer.Address.City}");


Blueprint E3_FlatDto → E3_Order:
  Id                        ←  ExactNameConvention   (Id)
  Customer                  ←  UnflatteningConvention  ((unflattened))

Result: Id=1, Customer.Name=Grace, Customer.Address.City=Arlington


## 4. `PrefixDropping` + `CaseConvention` — common entity↔DTO aliases

- `PrefixDropping` strips a recognised prefix (`db`, `tbl_`, `m_`) before matching.
- `CaseConvention` normalises snake_case / PascalCase / camelCase / kebab-case before matching.

Both are lower-confidence than `ExactName` so they only fire when the exact match fails.


In [6]:
public sealed class E4_DbRow   { public int dbUserId { get; init; } public string user_name { get; init; } = ""; public string EmailAddress { get; init; } = ""; }
public sealed class E4_UserDto { public int UserId { get; set; } public string UserName { get; set; } = ""; public string EmailAddress { get; set; } = ""; }

var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<E4_DbRow, E4_UserDto>(_ => { }))
    .Forge();

PrintLinks(sculptor);

var row = new E4_DbRow { dbUserId = 7, user_name = "ada", EmailAddress = "ada@example.com" };
var dto = sculptor.Map<E4_DbRow, E4_UserDto>(row);
Console.WriteLine($"\nResult: UserId={dto.UserId}, UserName={dto.UserName}, EmailAddress={dto.EmailAddress}");


Blueprint E4_DbRow → E4_UserDto:
  UserId                    ←  None                  ()
  UserName                  ←  CaseConvention        (user_name)
  EmailAddress              ←  ExactNameConvention   (EmailAddress)

Result: UserId=0, UserName=ada, EmailAddress=ada@example.com


## 5. `MethodToProperty` — `GetFullName()` → `FullName`

Parameterless `Get*` methods on the origin map against same-suffixed properties on the target. Useful for legacy getter-style code.


In [7]:
public sealed class E5_Person
{
    public string First { get; init; } = "";
    public string Last  { get; init; } = "";
    public string GetFullName() => $"{First} {Last}";
}

public sealed class E5_PersonDto
{
    public string First    { get; set; } = "";
    public string Last     { get; set; } = "";
    public string FullName { get; set; } = "";
}

var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<E5_Person, E5_PersonDto>(_ => { }))
    .Forge();

PrintLinks(sculptor);

var dto = sculptor.Map<E5_Person, E5_PersonDto>(new E5_Person { First = "Grace", Last = "Hopper" });
Console.WriteLine($"\nResult: FullName=\"{dto.FullName}\"");


Blueprint E5_Person → E5_PersonDto:
  First                     ←  ExactNameConvention   (First)
  Last                      ←  ExactNameConvention   (Last)
  FullName                  ←  MethodToPropertyConvention  (GetFullName())

Result: FullName="Grace Hopper"


## Next

- **`sprint-03-type-transformers.ipynb`** — once a link is established, how is the origin value *converted* to the target member type? That's where the type-transformer registry earns its keep.
